# Module 3 · Distributed Training & Multi-GPU for Deep Learning

**Phase:** Deep Learning Foundations  
**Prerequisites:** NB14 (PyTorch Foundations), NB15_06 (Transformers)  
**Toolchain:** PyTorch DDP · FSDP · DeepSpeed · accelerate  
**Objective:** Master the paradigms of distributed deep learning. Learn how to train models that exceed the VRAM of a single GPU by employing Data Parallelism, Fully Sharded Data Parallel (FSDP), and DeepSpeed ZeRO, along with practical multi-node workflows.

---

## Why Distributed Training is Mandatory in 2026

A 7B parameter LLM requires ~16GB of VRAM just to store its FP16 weights. During training, you also need memory for gradients, optimizer states (Adam needs 2x the model size), and activations. Training an 8B model requires roughly **120GB+ of VRAM** — far beyond any single consumer GPU, and exceeding even a single A100/H100 (80GB).

You cannot train or fine-tune modern models without distributed processing. 

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors rely on `DataParallel` or simple `.to('cuda')` and hit `RuntimeError: CUDA out of memory` instantly. Seniors understand how memory is consumed (Weights + Gradients + Optimizer + Activations) and deliberately partition the computation across devices using FSDP or DeepSpeed ZeRO.

**Mental Model:** 
- **Data Parallelism (DDP):** Every GPU has a copy of the whole model. They split the data batch.
- **Pipeline Parallelism:** GPU 1 has layers 1-10, GPU 2 has layers 11-20. Data flows through them sequentially.
- **Tensor Parallelism:** GPU 1 computes the left half of a matrix multiplication, GPU 2 computes the right half.
- **ZeRO/FSDP:** A magic trick where the model is sharded across GPUs, but each GPU gathers the pieces it needs *just in time* for computation, then discards them.

**Common Junior Pitfall:** Using `torch.nn.DataParallel` (DP) instead of `DistributedDataParallel` (DDP). DP uses a single PyTorch process and Python multi-threading, causing massive GIL contention and imbalanced GPU utilization. DDP spawns separate processes and is the standard.

---


---
## 4 · Parallelism Taxonomies (Data, Pipeline, Tensor)

Modern LLM training requires mixing different forms of parallelism, often called **3D Parallelism**. You must understand how they differ structurally.

### 1. Data Parallelism (DP/DDP)
Every GPU holds a full replica of the model. The dataset batch is split up. Gradients are synchronized at the end.

```mermaid
graph TD
    B[Batch 1024] --> S1[GPU 1: Batch 512]
    B --> S2[GPU 2: Batch 512]
    S1 --> M1[Full Model Replica]
    S2 --> M2[Full Model Replica]
    M1 --> G1[Gradients]
    M2 --> G2[Gradients]
    G1 <-->|All-Reduce| G2
```

### 2. Pipeline Parallelism (PP)
The model is split vertically by layers. GPU 1 gets exactly layers 1-10. GPU 2 gets layers 11-20. The batch flows sequentially through the GPUs.

```mermaid
graph LR
    Data --> G1[GPU 1: Layers 1-4] --> G2[GPU 2: Layers 5-8] --> G3[GPU 3: Layers 9-12] --> Loss
```
**Problem:** Pipeline bubbling. GPU 3 is completely idle while waiting for GPU 1 and 2 to finish the forward pass. Solutions like GPipe partition the micro-batch to minimize this bubble.

### 3. Tensor Parallelism (TP)
The model is split horizontally within the mathematical operations themselves. A single matrix multiplication ($Y = X \cdot W$) is split. GPU 1 computes $Y_{left} = X \cdot W_{left}$, GPU 2 computes $Y_{right} = X \cdot W_{right}$.

Because this requires synchronizing the output of almost every layer, Tensor Parallelism requires massive bandwidth and is almost exclusively used *within* a single node (over ultra-fast NVLink), never across nodes over a network.

> **Production Note — 3D Parallelism:** GPT-3 (175B) was trained using 8-way Tensor Parallelism (1 node), 64-way Pipeline Parallelism, and 6-way Data Parallelism.


---
## 5 · GPU Communication Patterns

When GPUs collaborate, they use standardized communication primitives provided by libraries like NCCL (Nvidia Collective Communication Library).

| Primitive | What it Does | Use Case |
|-----------|--------------|----------|
| **Broadcast** | GPU 0 sends a tensor to all other GPUs. | Broadcasting initial weights so everyone starts identically. |
| **Scatter** | GPU 0 chops a tensor into N pieces, sending piece `i` to GPU `i`. | Distributing a data batch. |
| **Gather** | All GPUs send their tensors to GPU 0, which concatenates them. | Collecting predictions for loss calculation. |
| **All-Gather** | Every GPU sends its tensor to *every other* GPU. Everyone ends up with the full concatenated tensor. | ZeRO-3 fetching sharded weights to reconstruct a layer JIT. |
| **Reduce** | All GPUs send their tensors to GPU 0, which applies an operation (e.g., sum, average). | Summing loss metrics. |
| **All-Reduce** | All GPUs apply an operation (e.g., sum) and *everyone* receives the final result. | Standard DDP gradient synchronization. |
| **Reduce-Scatter** | Performs a reduction (sum), but GPU `i` only gets the `i`-th chunk of the final result. | ZeRO-1 accumulating gradients but keeping the result sharded. |

**Why this matters:** Communication over the PCI-e bus or Ethernet is orders of magnitude slower than doing math on the GPU's ALU. Advanced distributed training is entirely an exercise in hiding communication latency behind computation.


---
## 6 · Fully Sharded Data Parallel (FSDP) & DeepSpeed ZeRO

What if the model is so large it does **not fit** on one GPU?
DDP fails here. If the model is 20GB and your GPU is 16GB, DDP instantly OOMs.

Enter **ZeRO (Zero Redundancy Optimizer)** developed by Microsoft DeepSpeed, and PyTorch's native implementation **FSDP**.

### The ZeRO Stages

ZeRO systematically removes communication and memory redundancies by sharding the three heavy components across GPUs, rather than replicating them.

| ZeRO Stage | What gets Sharded across GPUs | Memory Reduction |
|-----|-------------------------|------------------|
| **Standard DDP** | Nothing | No reduction |
| **ZeRO-1** | Optimizer States | ~4x reduction |
| **ZeRO-2** | Optimizer + Gradients | ~8x reduction |
| **ZeRO-3 (FSDP Full)** | Optimizer + Gradients + Model Weights | Scales linearly with GPUs |

**How ZeRO-3 / FSDP works:**
If you have 4 GPUs, each GPU holds only 25% of the model weights. The full model is never materialized in a single GPU's memory.
During the forward pass for Layer 1, GPU-A asks GPUs B, C, D for their pieces of Layer 1 (via an `All-Gather` operation). GPU-A temporarily rebuilds Layer 1 in its fast SRAM, computes the pass, and then immediately discards the borrowed pieces to free memory. It is purely "Just-In-Time" weight gathering.

### DeepSpeed Config Example

DeepSpeed is often configured via a JSON file rather than code:
```json
{
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": true
        },
        "offload_param": {
            "device": "cpu",
            "pin_memory": true
        },
        "overlap_comm": true,
        "contiguous_gradients": true
    }
}
```
By setting `offload_optimizer` to CPU, you use CPU RAM (which is cheap and plentiful, e.g., 1TB) to store the massive Adam momentum states, freeing up expensive VRAM exclusively for the model weights and activations.


In [ ]:
# Cell 0 — Setup
!pip install -q torch accelerate deepspeed
import torch
import torch.nn as nn
import os
print(f"PyTorch: {torch.__version__}")
print(f"Available GPUs: {torch.cuda.device_count()}")


---
## 1 · The Memory Math of Deep Learning

To understand why distributed training exists, you must understand exactly where your VRAM goes.

For a model with $P$ parameters trained in mixed precision (FP16/BF16) with Adam optimizer:

| Component | Memory Formula | 7B Model Example |
|-----------|---------------|------------------|
| **Model Weights** | $2P$ bytes (FP16) | ~14 GB |
| **Gradients** | $2P$ bytes (FP16) | ~14 GB |
| **Optimizer States** | $8P$ bytes (Adam keeps FP32 momentum + var) | ~56 GB |
| **Activations** | Depends on batch size x seq_len x depth | ~20-50 GB |
| **Total** | $\sim 12P + \text{activations}$ | **~104 GB+** |

An 80GB A100 isn't enough to train a 7B model locally. We must split it up.


---
## 2 · Gradient Accumulation: Poor Man's Distributed Training

If you only have one GPU and your model fits, but your desired *batch size* does not fit, you don't need multiple GPUs. You need **Gradient Accumulation**.

Instead of computing gradients and updating weights immediately, you compute gradients on small micro-batches, sum them up, and then update the weights once.


In [ ]:
# Cell 1 — Gradient Accumulation Implementation

def train_with_grad_accum(model, dataloader, optimizer, accumulation_steps=4):
    model.train()
    optimizer.zero_grad()
    
    for i, (inputs, targets) in enumerate(dataloader):
        # 1. Forward pass
        outputs = model(inputs)
        loss = torch.nn.functional.cross_entropy(outputs, targets)
        
        # 2. Scale the loss because we're averaging gradients over multiple steps
        loss = loss / accumulation_steps
        
        # 3. Backward pass (accumulates gradients, does NOT overwrite them)
        loss.backward()
        
        # 4. Step optimizer only every `accumulation_steps`
        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(dataloader):
            optimizer.step()
            optimizer.zero_grad()  # Clear gradients for next accumulation cycle

print("Math check: Loss is scaled by (1/accumulation_steps) because loss.backward() summates gradients.")


---
## 3 · Distributed Data Parallel (DDP)

When your model *fits* on one GPU, but you want to train *faster* by using multiple GPUs concurrently, you use **DDP (Distributed Data Parallel)**.

**How it works:**
1. PyTorch spawns one independent process per GPU.
2. Each GPU holds an identical copy of the model.
3. The dataset is split (distributed sampler), so each GPU processes a different slice of the batch.
4. During the backward pass, PyTorch intercepts the gradients and initiates an **All-Reduce** operation across the network (NVLink/PCIe).
5. Every GPU perfectly averages its gradients with all other GPUs.
6. Every GPU updates its weights identically. They remain synchronized.


In [ ]:
# Cell 2 — The anatomy of a DDP Script
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

# Note: This is an illustrative script. DDP requires running via `torchrun` on the CLI.
def ddp_setup():
    if 'LOCAL_RANK' in os.environ:
        dist.init_process_group("nccl") # NCCL is Nvidia's highly optimized GPU communication backend
        local_rank = int(os.environ["LOCAL_RANK"])
        torch.cuda.set_device(local_rank)
        return local_rank
    return 0

print("Typical DDP Execution command: torchrun --nproc_per_node=4 train_script.py")


---
## 4 · Fully Sharded Data Parallel (FSDP) & DeepSpeed ZeRO

What if the model is so large it does **not fit** on one GPU?
DDP fails here. If the model is 20GB and your GPU is 16GB, DDP instantly OOMs.

Enter **ZeRO (Zero Redundancy Optimizer)** developed by Microsoft DeepSpeed, and PyTorch's native implementation **FSDP**.

| ZeRO Stage | What gets Sharded across GPUs | Memory Reduction |
|-----|-------------------------|------------------|
| **Standard DDP** | Nothing | No reduction |
| **ZeRO-1** | Optimizer States | ~4x reduction |
| **ZeRO-2** | Optimizer + Gradients | ~8x reduction |
| **ZeRO-3 (FSDP Full)** | Optimizer + Gradients + Model Weights | Scales linearly with GPUs |

**How ZeRO-3 / FSDP works:**
If you have 4 GPUs, each GPU holds only 25% of the model weights.
During the forward pass for Layer 1, GPU-A asks GPUs B, C, D for their pieces of Layer 1. GPU-A temporarily rebuilds Layer 1 in its fast SRAM, computes the pass, and then immediately discards the borrowed pieces to free memory. It is purely "Just-In-Time" weight gathering.


In [ ]:
# Cell 3 — Applying PyTorch FSDP
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
import functools

def setup_fsdp_model(model, rank):
    if not dist.is_initialized():
        return model
        
    my_auto_wrap_policy = functools.partial(
        size_based_auto_wrap_policy, min_num_params=1e8
    )

    fsdp_model = FSDP(
        model,
        auto_wrap_policy=my_auto_wrap_policy,
        device_id=rank
    )
    return fsdp_model
    
print("FSDP allows training 70B parameter models (140GB) on 8x 40GB GPUs.")


---
## 7 · Multi-Node Networking & Orchestration

Training models larger than 70B parameters requires scaling beyond a single 8-GPU node. This introduces networking bottlenecks.

### The Networking Hierarchy

1. **Intra-Node (NVLink/NVSwitch):** 600 - 900 GB/s bandwidth. This is used for Tensor Parallelism (TP) which requires synchronization at every matrix multiplication.
2. **Inter-Node (InfiniBand / RoCE v2):** 25 - 50 GB/s bandwidth. Orders of magnitude slower. Because latency is higher, operations spanning nodes must be "coarse-grained" like Pipeline Parallelism (passing activations between layers) or Data Parallelism (syncing gradients once per step).

### Production Checklist: Multi-Node Orchestration

When spinning up a cluster of 32x H100 nodes, you do not SSH into each one and run `torchrun`. You use orchestration software.

- [x] **Slurm (Standard in HPC):** The traditional high-performance computing job scheduler. You define a bash script requesting `N` nodes, `G` GPUs, and the master IP. Slurm ensures all nodes launch the PyTorch script simultaneously.
- [x] **Kubernetes / Ray (Standard in Cloud):** Modern MLOps setups use tools like KubeFlow `PyTorchJob` or Ray Train. Ray handles restarting the entire training job automatically if one node experiences a hardware failure (which is common across thousands of GPUs).
- [x] **NCCL Backend:** Ensure PyTorch is compiled with NVIDIA's NCCL Backend (`dist.init_process_group("nccl")`). The older `gloo` backend is strictly for CPU-only arrays.
- [x] **InfiniBand (IB) Verification:** Run `nccl-tests` (like `all_reduce_perf`) before your Python script starts to verify that nodes are actually communicating over the optical IB switches and haven't fallen back to a slow 10Gbps Ethernet interface.


---
## 8 · Hands-off Distributed with Hugging Face Accelerate

Writing custom DDP and FSDP logic is error-prone. Hugging Face `accelerate` is effectively the standard wrapper for PyTorch distributed training inside modern codebases.

With accelerate, you write single-GPU code:


In [ ]:
# Cell 4 — HF Accelerate Pattern
"""
from accelerate import Accelerator

accelerator = Accelerator()

model = LlamaForCausalLM(...)
optimizer = torch.optim.AdamW(model.parameters())
dataloader = DataLoader(dataset)

model, optimizer, dataloader = accelerator.prepare(
    model, optimizer, dataloader
)

for inputs, targets in dataloader:
    outputs = model(inputs)
    loss = F.cross_entropy(outputs, targets)
    
    # Replaces loss.backward()
    accelerator.backward(loss)
    
    optimizer.step()
    optimizer.zero_grad()
"""
print("Hugging Face Accelerate eliminates 90% of boilerplate distributed code.")


---
## ✅ Knowledge Check

### Q1: Memory Math
<details><summary>Why does the optimizer state dominate VRAM?</summary>
Adam maintains FP32 moving averages (momentum and variance) for safety, requiring 8 bytes per parameter regardless of if weights are FP16/BF16.
</details>

### Q2: ZeRO Offloading
<details><summary>Why is ZeRO CPU offloading useful but slow?</summary>
It pushes the massive optimizer states to system RAM (1TB+ capacity), freeing up VRAM. However, at every step, data must cross the slow PCIe bus (max 64GB/s) to return to the GPU for computation, bottlenecking training speed.
</details>

### Q3: Parallelism Types
<details><summary>Which form of parallelism requires the highest networking bandwidth?</summary>
Tensor Parallelism. Because it splits the actual matrix multiplication, it requires an All-Reduce synchronization *inside* the layer, meaning it must use intra-node NVLink.
</details>

---
## 🔨 Practical Practice

1. Estimate the VRAM required to finetune a 13B model in FP16 with Adam using standard DDP.
2. Diagram the flow of data through a 4-node cluster implementing 4-way Pipeline Parallelism and 8-way Tensor Parallelism.
3. Set up a DeepSpeed config JSON that turns on ZeRO Stage 3 with contiguous memory gradients and overlap communication.

---
## 🎯 Summary & Key Takeaways

| Strategy | Purpose | When to Use |
|-----------|------|--------|
| **Gradient Accum.** | Bypass batch-size VRAM limits | 1 GPU, large models/batches |
| **DDP** | Train faster (parallel processing) | Multi-GPU, local model fits in 1 GPU |
| **Pipeline / Tensor** | Spatial splitting | Model > 1 GPU VRAM |
| **FSDP / ZeRO-3** | Algorithmic Just-In-Time Gathering | Multi-node, massive models |
| **Accelerate** | Write distributed code easily | Always |

**Next →** `16_01_nlp_foundations.ipynb` — Modern NLP Fundamentals
